In [1]:
# Cell 1: Check GPU status
!nvidia-smi

Tue Apr 15 01:38:39 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             49W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2: Install dependencies and import libraries
!pip install datasets
import os
import torch
from torch.utils.data import DataLoader
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          get_linear_schedule_with_warmup, DataCollatorForLanguageModeling)
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2024.12.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which

In [3]:
# Cell 3: Define directories and hyperparameters
PERSISTENT_DIR = "/content/drive/MyDrive/env"  # Base directory for saved files
MODEL_DIR = os.path.join(PERSISTENT_DIR, "model")  # Directory for base model & tokenizer
FINAL_OUTPUT_DIR = "/content/drive/MyDrive/fin"  # Dedicated folder for final fine-tuned model
TRAIN_DS_PATH = os.path.join(PERSISTENT_DIR, "train_dataset")
EVAL_DS_PATH = os.path.join(PERSISTENT_DIR, "eval_dataset")

NUM_EPOCHS = 1
BATCH_SIZE = 16
LEARNING_RATE = 1e-5
GRAD_ACC_STEPS = 1
MAX_LENGTH = 512
IGNORE_INDEX = -100
CHECKPOINT_FREQ = 1000

# Create directories if they don’t exist
os.makedirs(PERSISTENT_DIR, exist_ok=True)
os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)
torch.cuda.empty_cache()

In [4]:
# Cell 4: Load or prepare tokenized datasets
if os.path.exists(TRAIN_DS_PATH) and os.path.exists(EVAL_DS_PATH):
    print("Loading tokenized datasets from disk...")
    train_dataset = load_from_disk(TRAIN_DS_PATH)
    eval_dataset = load_from_disk(EVAL_DS_PATH)
else:
    print("Downloading the 'gyanai/ganita' dataset from Hugging Face...")
    dataset = load_dataset("gyanai/ganita")
    if "eval" not in dataset.keys():
        dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
        train_ds = dataset["train"]
        eval_ds = dataset["test"]
    else:
        train_ds = dataset["train"]
        eval_ds = dataset["eval"]

    # Format samples: combine question and answer
    def format_sample(sample):
        text = f"Question: {sample['question']}\nAnswer: {sample['answer']}"
        return {"text": text}

    train_ds = train_ds.map(format_sample, remove_columns=train_ds.column_names)
    eval_ds = eval_ds.map(format_sample, remove_columns=eval_ds.column_names)

    # Load tokenizer
    model_name = "prithivMLmods/Llama-3.2-3B-Math-Oct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    # Tokenize datasets
    def tokenize_function(sample):
        return tokenizer(sample["text"], truncation=True, max_length=MAX_LENGTH, padding="max_length")

    print("Tokenizing datasets...")
    train_ds = train_ds.map(tokenize_function, batched=True)
    eval_ds = eval_ds.map(tokenize_function, batched=True)

    # Save tokenized datasets
    train_ds.save_to_disk(TRAIN_DS_PATH)
    eval_ds.save_to_disk(EVAL_DS_PATH)
    train_dataset = load_from_disk(TRAIN_DS_PATH)
    eval_dataset = load_from_disk(EVAL_DS_PATH)

Loading tokenized datasets from disk...


In [5]:
# Cell 5: Load model and tokenizer
if os.path.exists(FINAL_OUTPUT_DIR) and os.listdir(FINAL_OUTPUT_DIR):
    print("Loading fine-tuned model and tokenizer from final output directory...")
    tokenizer = AutoTokenizer.from_pretrained(FINAL_OUTPUT_DIR)
    model = AutoModelForCausalLM.from_pretrained(FINAL_OUTPUT_DIR, torch_dtype=torch.float16).to("cuda")
else:
    if os.path.exists(MODEL_DIR):
        print("Loading base model and tokenizer from disk...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
        model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.float16).to("cuda")
    else:
        print("Downloading base model and tokenizer from Hugging Face Hub...")
        model_name = "prithivMLmods/Llama-3.2-3B-Math-Oct"
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16).to("cuda")
        os.makedirs(MODEL_DIR, exist_ok=True)
        model.save_pretrained(MODEL_DIR)
        tokenizer.save_pretrained(MODEL_DIR)

    # Memory optimizations
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

Loading fine-tuned model and tokenizer from final output directory...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
# Cell 6: Prepare datasets and data loaders
columns_to_keep = ["input_ids", "attention_mask"]
train_dataset = train_dataset.remove_columns([col for col in train_dataset.column_names if col not in columns_to_keep])
eval_dataset = eval_dataset.remove_columns([col for col in eval_dataset.column_names if col not in columns_to_keep])
train_dataset.set_format(type="torch", columns=columns_to_keep)
eval_dataset.set_format(type="torch", columns=columns_to_keep)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=8
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator,
    num_workers=8
)
print(f"Training dataset size: {len(train_dataset)}, steps: {len(train_loader)}")

Training dataset size: 485849, steps: 30366


In [7]:
# Cell 7: Define custom rotary embedding (optional, only if needed)
from transformers.models.llama.modeling_llama import LlamaRotaryEmbedding
class CustomLlamaRotaryEmbedding(LlamaRotaryEmbedding):
    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None):
        super().__init__(128, max_position_embeddings, base, device or torch.device("cuda"))
        self._set_cos_sin_cache(max_position_embeddings=max_position_embeddings,
                                device=device or torch.device("cuda"),
                                dtype=torch.float16)

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def custom_apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    head_dim = q.shape[-1]
    if cos.shape[-1] != head_dim:
        cos = cos[..., :head_dim]
        sin = sin[..., :head_dim]
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

# Apply patches only if training (not loading fine-tuned model)
if not os.path.exists(FINAL_OUTPUT_DIR) or not os.listdir(FINAL_OUTPUT_DIR):
    for module in model.modules():
        if isinstance(module, LlamaRotaryEmbedding):
            module.__class__ = CustomLlamaRotaryEmbedding
        if hasattr(module, "apply_rotary_pos_emb"):
            module.apply_rotary_pos_emb = custom_apply_rotary_pos_emb
    print("Rotary embedding patch applied.")
else:
    print("Skipping rotary embedding patch as fine-tuned model is loaded.")

print(f"Model config: hidden_size={model.config.hidden_size}, num_attention_heads={model.config.num_attention_heads}")

Skipping rotary embedding patch as fine-tuned model is loaded.
Model config: hidden_size=3072, num_attention_heads=24


In [8]:
# Cell 8: Optimizer and scheduler (only if training)
if not os.path.exists(FINAL_OUTPUT_DIR) or not os.listdir(FINAL_OUTPUT_DIR):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = (len(train_loader) // GRAD_ACC_STEPS) * NUM_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)

In [9]:
# Cell 9: Subsample dataset (only if training)
if not os.path.exists(FINAL_OUTPUT_DIR) or not os.listdir(FINAL_OUTPUT_DIR):
    train_dataset = train_dataset.shuffle(seed=42).select(range(16000))
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator, num_workers=8)
    total_steps = (len(train_loader) // GRAD_ACC_STEPS) * NUM_EPOCHS
    print(f"Subsampled training dataset size: {len(train_dataset)}, steps: {len(train_loader)}")

In [10]:
# Cell 10: Training loop (runs only if model isn’t fine-tuned yet)
if not os.path.exists(FINAL_OUTPUT_DIR) or not os.listdir(FINAL_OUTPUT_DIR):
    global_step = 0
    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for step, batch in enumerate(progress_bar):
            batch = {k: v.to("cuda") for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            epoch_loss += loss.item()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            progress_bar.set_postfix(loss=loss.item())

            if (step + 1) % CHECKPOINT_FREQ == 0:
                checkpoint_path = os.path.join(FINAL_OUTPUT_DIR, f"step_{global_step}")
                model.save_pretrained(checkpoint_path)
                tokenizer.save_pretrained(checkpoint_path)
                print(f"Checkpoint saved at step {global_step} to {checkpoint_path}")

            torch.cuda.empty_cache()

        # Save after each epoch
        epoch_save_path = os.path.join(FINAL_OUTPUT_DIR, f"epoch_{epoch+1}")
        model.save_pretrained(epoch_save_path)
        tokenizer.save_pretrained(epoch_save_path)
        print(f"Epoch {epoch+1} saved to {epoch_save_path}")

    # Final save after training completes
    model.save_pretrained(FINAL_OUTPUT_DIR)
    tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
    print(f"Final fine-tuned model saved to {FINAL_OUTPUT_DIR}")
else:
    print("Training skipped: Fine-tuned model already exists.")

Training skipped: Fine-tuned model already exists.


In [11]:
def debug_shapes():
    for batch in train_loader:
        input_ids_shape = batch["input_ids"].shape
        attention_mask_shape = batch["attention_mask"].shape
        print("input_ids shape:", input_ids_shape)
        print("attention_mask shape:", attention_mask_shape)
        if input_ids_shape != attention_mask_shape:
            print("[ERROR] input_ids and attention_mask shapes mismatch!")
        else:
            print("[OK] input_ids and attention_mask shapes match.")
        break
debug_shapes()

input_ids shape: torch.Size([16, 256])
attention_mask shape: torch.Size([16, 256])
[OK] input_ids and attention_mask shapes match.


w/o text parsing

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [15]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# --- Configuration ---
# Ensure MODEL_DIR is correctly defined from your previous cells
# Example: MODEL_DIR = "/content/drive/MyDrive/env/model"
# Ensure torch is imported

print("--- Loading base model and tokenizer for evaluation ---")

# --- Load Model and Tokenizer from MODEL_DIR ---
# Check if the base model directory exists
if os.path.exists(MODEL_DIR):
    # Load tokenizer from the specified directory
    eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

    # Load the base model from the specified directory
    # Using float16 for efficiency, ensure GPU has enough memory
    eval_model = AutoModelForCausalLM.from_pretrained(
        MODEL_DIR,
        torch_dtype=torch.float16
    ).to("cuda") # Move model to GPU

    # Set the model to evaluation mode (disables dropout, etc.)
    eval_model.eval()

    # --- Configure Padding Tokens ---
    # Set the tokenizer's pad token to its EOS token
    eval_tokenizer.pad_token = eval_tokenizer.eos_token
    # Align the model's pad token ID config with the tokenizer
    eval_model.config.pad_token_id = eval_tokenizer.eos_token_id

    print("Base model and tokenizer loaded successfully and moved to GPU.")

else:
    # Handle case where the base model directory wasn't found
    print(f"ERROR: Base model directory not found at '{MODEL_DIR}'.")
    print("Cannot load model for evaluation. Please ensure MODEL_DIR is correct.")
    # Set variables to None to prevent errors later if loading failed
    eval_model = None
    eval_tokenizer = None

--- Loading base model and tokenizer for evaluation ---


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model and tokenizer loaded successfully and moved to GPU.


In [23]:
print(eval_model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((3072,), eps=1e

In [21]:
def solve_math_problem(model, tokenizer, question, max_length):
    """
    Generates an answer using the provided model and tokenizer instance.
    Assumes model is already on the correct device and in eval mode.
    """
    # --- Input Validation (Optional but recommended) ---
    if not model or not tokenizer:
        return "[Error: Model or Tokenizer not provided]"

    try:
        # --- Prepare Input ---
        # Truncate question string if necessary before creating prompt
        # Ensure max_length is an integer
        max_len_int = int(max_length)
        question = question[:max_len_int]
        prompt = f"Question: {question}\nAnswer:"

        # Tokenize the prompt using the provided tokenizer
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_len_int,
            padding=True # Pad to max_length
        )

        # Move tokenized inputs to the same device as the model
        input_ids = inputs["input_ids"].to(model.device)
        attention_mask = inputs["attention_mask"].to(model.device)

        # --- Generate Output ---
        with torch.no_grad(): # Disable gradient calculations
            # Use the provided model instance for generation
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=200,  # Max tokens for the answer part
                num_beams=1,         # Use greedy search (efficient and robust)
                do_sample=False,     # Disable sampling
                early_stopping=True  # Stop if EOS token is generated
            )

        # --- Decode and Extract Answer ---
        # Decode the generated token IDs back into text
        full_decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Remove the prompt part to isolate the answer
        if full_decoded_output.startswith(prompt):
             answer_only = full_decoded_output[len(prompt):].strip()
        else:
             # Fallback logic for removing prompt if needed
             question_part = prompt.split("\nAnswer:")[0].strip()
             temp_answer = full_decoded_output.replace(question_part, "").strip()
             if temp_answer.lower().startswith("answer:"):
                 answer_only = temp_answer[len("answer:"):].strip()
             else:
                 answer_only = temp_answer

        return answer_only

    except Exception as e:
        # Generic error message for robustness
        # Consider logging 'e' if needed for debugging without printing
        # print(f"Error in solve_math_problem: {e}") # Uncomment for debugging
        return "[Error processing request]"

In [25]:
# Cell 12: Interactive inference with step logging
if eval_model and eval_tokenizer:

    # Get input from the user
    question = input("Enter any math word problem:\n")

    # Optional: Clean the input (like the ASCII encoding used before)
    # question = question.encode('ascii', 'ignore').decode('ascii')

    print("\nThinking..............")

    # Call the solve_math_problem function, passing the loaded model, tokenizer,
    # the user's question, and the defined MAX_LENGTH.
    answer = solve_math_problem(
        eval_model,
        eval_tokenizer,
        question,
        MAX_LENGTH
    )

    # Print the generated answer
    print("\nAnswer:")
    print(answer)

else:
    # Handle the case where the model/tokenizer didn't load correctly
    print("Model or tokenizer not loaded. Cannot run inference.")
    print("Please ensure the previous cell executed successfully and MODEL_DIR is correct.")

Enter any math word problem:
A cooler is filled with 24 cans of cherry soda and orange pop. If there are twice as many cans of orange pop as there are of cherry soda, how many cherry sodas are there?

Thinking..............


/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:679: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(



Answer:
Let's denote the number of cherry sodas as C and the number of orange pops as O. We are given that there are twice as many cans of orange pop as there are of cherry soda, so we can write the equation:
O = 2C
We are also given that the total number of cans is 24, so we can write another equation:
C + O = 24
Now we can substitute the first equation into the second equation to get:
C + 2C = 24
Combine like terms:
3C = 24
Now, divide both sides by 3 to solve for C:
C = 24 / 3
C = 8
So, there are 8 cans of cherry soda. The answer is $\boxed{8}$.


Evaluation

Tabular rep

In [31]:
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, load_from_disk # Assuming these are needed if not imported earlier
from tabulate import tabulate
import random
from tqdm import tqdm
import re # Ensure re is imported for extract_numeric_answer

# --- Prerequisite Variables (Ensure these are defined and loaded earlier) ---
# eval_model: Your loaded base model instance on GPU, in eval mode.
# eval_tokenizer: Your loaded base tokenizer instance.
# MAX_LENGTH: The maximum sequence length (e.g., 512).
# PERSISTENT_DIR: Path to your persistent storage.
# solve_math_problem: The function defined as solve_math_problem(model, tokenizer, question, max_length)
# extract_numeric_answer: The function defined to extract numbers.

# --- Load Raw Evaluation Dataset ---
RAW_EVAL_DS_PATH = os.path.join(PERSISTENT_DIR, "raw_eval_dataset")
if os.path.exists(RAW_EVAL_DS_PATH):
    print(f"Loading raw evaluation dataset from {RAW_EVAL_DS_PATH}...")
    eval_dataset = load_from_disk(RAW_EVAL_DS_PATH)
else:
    # Fallback logic if raw dataset isn't saved (adjust as needed)
    print("Raw evaluation dataset not found. Attempting to download...")
    # This part might need adjustment based on how you handle dataset loading
    try:
        dataset = load_dataset("gyanai/ganita") # Or your specific dataset
        if "eval" not in dataset.keys():
             # Example split if only train is available
             dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
             eval_dataset = dataset["test"]
        else:
             eval_dataset = dataset["eval"]
        # Ensure directory exists before saving
        os.makedirs(PERSISTENT_DIR, exist_ok=True)
        eval_dataset.save_to_disk(RAW_EVAL_DS_PATH)
        print(f"Raw evaluation dataset saved to {RAW_EVAL_DS_PATH}")
    except Exception as e:
        print(f"Failed to load or download evaluation dataset: {e}")
        # Set eval_dataset to None or an empty list to prevent errors below
        eval_dataset = None

# --- Evaluation Setup ---
if eval_dataset:
    # Select 100 random samples initially (or fewer if dataset is small)
    total_samples = len(eval_dataset)
    sample_size = min(100, total_samples) # Process up to 100 samples
    random.seed(42)  # For reproducibility
    random_indices = random.sample(range(total_samples), sample_size)
    eval_subset = eval_dataset.select(random_indices)
    print(f"Selected {len(eval_subset)} random samples for processing.")

    # List to store results ONLY for matching answers, limit to 10
    matching_results = []
    max_display_matches = 10 # Display up to 10 matches

    # --- Evaluation Loop ---
    print(f"\nStarting evaluation (processing up to {sample_size} samples)...")
    for i, sample in enumerate(tqdm(eval_subset, desc="Evaluating Samples")):
        # Stop processing if we already found the desired number of matches
        if len(matching_results) >= max_display_matches:
            break

        question = sample["question"]
        true_answer_text = sample["answer"]

        # --- Generate model's answer ---
        # Check if model and tokenizer are loaded before calling
        if eval_model and eval_tokenizer:
            generated_answer = solve_math_problem(
                eval_model,      # Pass the pre-loaded model
                eval_tokenizer,  # Pass the pre-loaded tokenizer
                question,
                MAX_LENGTH       # Pass the defined max length
            )
        else:
            generated_answer = "[Error: Model not loaded]"
            # Skip processing this sample if model isn't available
            print(f"\nSkipping sample due to model loading issue.")
            continue # Go to the next sample

        # --- Extract numeric answers ---
        generated_numeric = extract_numeric_answer(generated_answer)
        true_numeric = extract_numeric_answer(true_answer_text)

        # --- Compare Answers ---
        # Check if extraction was successful and if answers match
        if generated_numeric is not None and true_numeric is not None:
            # Check for match with tolerance for floating-point precision
            exact_match = abs(generated_numeric - true_numeric) < 1e-6
            if exact_match:
                # Only store data if it's a match and we need more results
                if len(matching_results) < max_display_matches:
                     match_str = "Yes" # Store "Yes" for the table
                     matching_results.append([
                        question,
                        generated_numeric, # Already checked not None
                        true_numeric,      # Already checked not None
                        match_str
                     ])
        # --- No action needed if it's not a match or extraction failed ---

    # --- Prepare Data for Display Table ---
    data_for_table = []
    num_displayed = len(matching_results) # Actual number of matches found (up to 10)

    for idx, result_row in enumerate(matching_results):
        data_for_table.append(result_row)
        # Add separator if not the last item
        if idx < num_displayed - 1:
             data_for_table.append(["-" * 25, "-" * 10, "-" * 10, "-" * 5]) # Visual separator


    # --- Display Results ---
    if num_displayed > 0:
        # Create and display the formatted table using only the matching results
        headers = ["Question", "Generated Numeric", "Actual Numeric", "Exact Match"]
        table = tabulate(data_for_table, headers=headers, tablefmt="pretty", maxcolwidths=[50, None, None, None], showindex=False)
        print(f"\n--- Model Evaluation Results ({num_displayed} Samples Shown) ---")
        print(table)

        # Calculate and print accuracy based ONLY on the displayed samples
        # Since only matches are displayed, matches = num_displayed
        matches_displayed = num_displayed
        total_displayed = num_displayed
        accuracy_displayed = matches_displayed / total_displayed if total_displayed > 0 else 0
        print(f"\nAccuracy across {total_displayed} displayed samples: {accuracy_displayed:.2%} ({matches_displayed} out of {total_displayed} matches)")
    else:
        print(f"\nNo matching answers found within the first {sample_size} processed samples to display.")

else:
    print("\nEvaluation dataset not loaded. Skipping evaluation.")

Loading raw evaluation dataset from /content/drive/MyDrive/env/raw_eval_dataset...
Selected 100 random samples for processing.

Starting evaluation (processing up to 100 samples)...


Evaluating Samples:  29%|██▉       | 29/100 [03:10<07:45,  6.55s/it]


--- Model Evaluation Results (10 Samples Shown) ---
+----------------------------------------------------+-------------------+----------------+-------------+
|                      Question                      | Generated Numeric | Actual Numeric | Exact Match |
+----------------------------------------------------+-------------------+----------------+-------------+
| When three positive integers are divided by $12$,  |       10.0        |      10.0      |     Yes     |
|  the remainders are $7,$ $9,$ and X respectively.  |                   |                |             |
|  When the sum of the three integers is divided by  |                   |                |             |
|   $12$, The remainder is 2. What is the value of   |                   |                |             |
|                unknown variable X?                 |                   |                |             |
|             -------------------------              |    ----------     |   ----------   |    ----